# RS-Mamba TensorRT Export De-Risk (`selective_scan`) — Phase 4

**Goal of this notebook (investigative only):** find out *whether and how* `RSM_CD` crosses the
PyTorch → ONNX → TensorRT boundary. The known risk (root `CLAUDE.md`) is the custom
`selective_scan_cuda_oflex` CUDA op, which is **not** a standard ONNX/TensorRT operator.

We do **not** try to produce a working TensorRT engine here. The deliverable is:
1. The **exact** failure from a naive `torch.onnx.export` of the real (`v3`) model.
2. A **control** export of the `fake` (no-op scan) model to isolate whether the compiled scan
   is the *only* blocker, or whether `CrossScan`/`CrossMerge` also block.
3. A structured **finding** (§5) with the recommended path.

Reuses the validated Kaggle T4 env recipe from `model/CLAUDE.md`. Run top-to-bottom on a
Kaggle T4 GPU notebook with Internet enabled. Export attempts are caught and verified, so the
notebook completes without an uncaught exception even when an attempt fails.

## 1. Environment
Reuses the validated Kaggle recipe (see `model/CLAUDE.md`).

In [13]:
import os

WORK = "/kaggle/working"
os.chdir(WORK)

REPO_URL = "https://github.com/fergalriordan/Mamba-RS-Engine.git"
VMAMBA_URL = "https://github.com/MzeroMiko/VMamba.git"

# This repo carries the vendored RS-Mamba code (model/vendor/rs_mamba_cd).
if not os.path.isdir(f"{WORK}/Mamba-RS-Engine"):
    !git clone --depth 1 {REPO_URL}
# VMamba provides the selective-scan CUDA kernel.
if not os.path.isdir(f"{WORK}/VMamba"):
    !git clone --depth 1 {VMAMBA_URL}

VENDOR = f"{WORK}/Mamba-RS-Engine/model/vendor/rs_mamba_cd"
assert os.path.isdir(VENDOR), f"vendored code not found at {VENDOR}"
print("vendor OK:", VENDOR)

vendor OK: /kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd


In [12]:
# Install deps individually WITHOUT the numpy==1.22.4 / Pillow pins that fail to build on Py3.12.
# torch / torchvision / numpy / PIL / opencv / kagglehub come preinstalled on Kaggle.
# For export we additionally want onnx + onnxruntime; einops/timm/fvcore are model deps.
# onnxscript is REQUIRED by the dynamo (torch.export) ONNX backend — without it,
# `torch.onnx.export(..., dynamo=True)` dies at import before tracing the model.
!pip install -q einops timm fvcore onnx onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 15.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 13.6 MB/s eta 0:00:00


In [14]:
# Build the selective-scan CUDA kernel -> installs `selective_scan_cuda_oflex`.
# MUST run BEFORE importing rs_mamba_cd. setup.py compiles MODES=["oflex"].
%cd /kaggle/working/VMamba/kernels/selective_scan
!pip install -q .
%cd /kaggle/working

/kaggle/working/VMamba/kernels/selective_scan
  Preparing metadata (setup.py) ... done
/kaggle/working


## 2. Make vendored code importable, then load `RSM_CD` and confirm the forward pass

Import-ordering gotchas (from `model/CLAUDE.md`):
- only the vendor **root** on `sys.path` (so `utils` resolves as a package);
- `import torch` **before** `import selective_scan_cuda_oflex` (libc10 link);
- if `rs_mamba_cd` was imported before the kernel build, drop & reimport.

We do **not** need the dataset loader for export, so the albumentations `A.Flip` shim is
skipped — we only import `RSM_CD`.

In [16]:
import sys

VENDOR = "/kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd"

# Only the vendor ROOT goes on sys.path so `utils` resolves to the package dir.
bad = f"{VENDOR}/utils"
if bad in sys.path:
    sys.path.remove(bad)
if VENDOR not in sys.path:
    sys.path.insert(0, VENDOR)

# Clear any stale / half-imported modules from earlier attempts so they re-resolve.
for m in [k for k in list(sys.modules) if k == "utils" or k.startswith("utils.")]:
    del sys.modules[m]

# Import torch FIRST so its shared libs (libc10.so etc.) are loaded into the process.
import torch

# Then bind the kernel (must happen before importing rs_mamba_cd).
import selective_scan_cuda_oflex
print("kernel OK:", selective_scan_cuda_oflex.__file__)

if "rs_mamba_cd" in sys.modules:
    del sys.modules["rs_mamba_cd"]

from rs_mamba_cd import RSM_CD

import onnx, onnxruntime
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))
print("onnx", onnx.__version__, "| onnxruntime", onnxruntime.__version__)

kernel OK: /usr/local/lib/python3.12/dist-packages/selective_scan_cuda_oflex.cpython-312-x86_64-linux-gnu.so
torch 2.10.0+cu128 | cuda True | Tesla T4
onnx 1.21.0 | onnxruntime 1.27.0


/kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd/rs_mamba_cd.py:130: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd/rs_mamba_cd.py:158: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd/rs_mamba_cd.py:176: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd/rs_mamba_cd.py:184: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/kaggle/working/Mamb

In [17]:
device = "cuda"

def build_model(forward_type):
    """RSM-CD tiny (51.95M params), matching the Phase-3 baseline config."""
    return RSM_CD(
        dims=96, depths=[2, 2, 9, 2],
        ssm_d_state=16, ssm_ratio=2.0, ssm_dt_rank="auto",
        mlp_ratio=4.0, drop_path_rate=0.2,
        forward_type=forward_type,
    ).to(device).eval()

net = build_model("v3")
print(f"RSM-CD tiny (v3): {sum(p.numel() for p in net.parameters())/1e6:.2f}M params")

# Dummy bitemporal pair (before/after), batch 1, 3x256x256 each.
x1 = torch.randn(1, 3, 256, 256, device=device)
x2 = torch.randn(1, 3, 256, 256, device=device)

with torch.no_grad():
    y = net(x1, x2)
print("forward OK -> output shape:", tuple(y.shape))
assert tuple(y.shape) == (1, 1, 256, 256), "unexpected output shape"

RSM-CD tiny (v3): 51.95M params
forward OK -> output shape: (1, 1, 256, 256)


### The export boundary, from the source

All *real* `forward_type` values route the SSM through a compiled CUDA `torch.autograd.Function`
(`model/vendor/rs_mamba_cd/rs_mamba_cd.py`):

| `forward_type` | autograd.Function | compiled op call |
|---|---|---|
| `v3` (baseline) | `SelectiveScanOflex` | `selective_scan_cuda_oflex.fwd(...)` — line 201 |
| `v2`, `v0` | `SelectiveScanCore` | `selective_scan_cuda_core.fwd(...)` — line 179 |
| `v01` | `SelectiveScanMamba` | `selective_scan_cuda.fwd(...)` — line 153 |
| `fake` | `SelectiveScanFake` | **no-op** `out = u` — line 217 (traceable, but wrong math) |

The compiled ops have **no ONNX symbolic**, so the tracer cannot lower them. The scan is invoked
inside `cross_selective_scan` (line 460), with SSM inputs assembled at lines 513–536 — that's the
decomposition boundary a future fix would target. Note `CrossScan`/`CrossMerge` (lines 288/331)
are *also* custom autograd.Functions; the `fake` control in §4 tells us whether they block too.

## 3. Naive ONNX export of the real (`v3`) model

We attempt both exporter backends, because they handle custom `autograd.Function`s differently:
- **legacy TorchScript exporter** (`torch.onnx.export`, default) — with `autograd_inlining` it
  traces *into* the Function's forward and reaches the pybind C++ op `selective_scan_cuda_oflex.fwd`.
  ⚠️ It may **return without raising** even though no valid ONNX op exists for the scan — so a
  "SUCCESS" here is **not** conclusive and is verified in §4.5.
- **dynamo exporter** (`dynamo=True`) — uses `torch.export`/FX; it can fail at a different
  (deeper) point. Requires `onnxscript` (installed in §1).

Both are wrapped so the full traceback is printed verbatim and captured for the §5 finding.

In [19]:
import io, traceback, contextlib

# Captured results to summarise in the finding cell (§5).
results = {}

def attempt_export(model, args, fname, *, dynamo, input_names, output_names, dynamic_axes):
    """Run one export attempt; capture success/failure + full traceback. Never raises."""
    label = f"{'dynamo' if dynamo else 'torchscript'} -> {fname}"
    print(f"\n===== EXPORT ATTEMPT: {label} =====")
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            torch.onnx.export(
                model, args, fname,
                input_names=input_names, output_names=output_names,
                dynamic_axes=dynamic_axes, opset_version=17,
                dynamo=dynamo,
            )
        ok = os.path.isfile(fname)
        print(buf.getvalue()[-2000:])
        print("RESULT: SUCCESS" if ok else "RESULT: returned without writing a file")
        results[label] = {"ok": ok, "error": None}
    except Exception as e:
        tb = traceback.format_exc()
        # Surface the exporter's own captured logs too — the useful op name is often there.
        log_tail = buf.getvalue()[-3000:]
        if log_tail.strip():
            print("--- exporter log tail ---")
            print(log_tail)
        print("--- traceback ---")
        print(tb[-4000:])
        print(f"RESULT: FAILED -> {type(e).__name__}: {str(e)[:300]}")
        results[label] = {"ok": False, "error": f"{type(e).__name__}: {e}", "traceback": tb}
    return results[label]

In [20]:
EXPORT_KW = dict(
    input_names=["t1", "t2"],
    output_names=["change_logits"],
    dynamic_axes={"t1": {0: "batch"}, "t2": {0: "batch"}, "change_logits": {0: "batch"}},
)

# 3a. Legacy TorchScript exporter on the real v3 model.
attempt_export(net, (x1, x2), "rsm_cd_v3_torchscript.onnx", dynamo=False, **EXPORT_KW)

# 3b. Dynamo exporter on the real v3 model.
attempt_export(net, (x1, x2), "rsm_cd_v3_dynamo.onnx", dynamo=True, **EXPORT_KW)


===== EXPORT ATTEMPT: torchscript -> rsm_cd_v3_torchscript.onnx =====
/tmp/ipykernel_58/664370153.py:13: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(

RESULT: SUCCESS

===== EXPORT ATTEMPT: dynamo -> rsm_cd_v3_dynamo.onnx =====


W0628 14:44:50.400000 58 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0628 14:44:51.508000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0628 14:44:51.510000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int

--- exporter log tail ---
/tmp/ipykernel_58/664370153.py:13: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
[torch.onnx] Obtain model graph for `RSM_CD([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `RSM_CD([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `RSM_CD([...]` with `torch.export.export(..., strict=True)`...
[torch.onnx] Obtain model graph for `RSM_CD([...]` with `torch.export.export(..., strict=True)`... ❌

--- traceback ---
ules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd/rs_mamba_cd.py", line 873, in forward
    y = self.forward_core(x, channel_first=with_dconv)
       

{'ok': False,
 'error': "TorchExportError: Failed to export the model with torch.export. \x1bThis is step 1/3\x1b of exporting the model to ONNX. Next steps:\n- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.\n- Debug `torch.export.export` and submit a PR to PyTorch.\n- Create an issue in the PyTorch GitHub repository against the \x1b*torch.export*\x1b component and attach the full error stack as well as reproduction scripts.\n\n## Exception summary\n\n<class 'RuntimeError'>: Cannot access data pointer of Tensor (e.g. FakeTensor, FunctionalTensor). If you're using torch.compile/export/fx, it is likely that we are erroneously tracing into a custom kernel. To fix this, please wrap the custom kernel into an opaque custom op. Please see the following for details: https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html\n\n(Refer to the full stack trace above for more informat

## 4. Control — does *everything except the scan* export?

Build the same architecture with `forward_type="fake"`. `SelectiveScanFake` is a no-op
(returns its input `u`), so it contains **no compiled op** and is traceable. The math is wrong,
but that's irrelevant here — we only care whether the graph *exports*.

Interpretation:
- **`fake` exports cleanly** → the compiled scan is the **sole** blocker; `CrossScan`/`CrossMerge`
  trace fine. Favors a *targeted* fix (plugin or decompose of just the scan).
- **`fake` also fails** → there are additional autograd.Function blockers; the decompose surface
  is larger. The traceback will name which op.

In [21]:
net_fake = build_model("fake")
with torch.no_grad():
    y_fake = net_fake(x1, x2)
print("fake forward OK -> output shape:", tuple(y_fake.shape))

# 4a. Legacy TorchScript exporter on the fake model.
attempt_export(net_fake, (x1, x2), "rsm_cd_fake_torchscript.onnx", dynamo=False, **EXPORT_KW)

# 4b. Dynamo exporter on the fake model.
attempt_export(net_fake, (x1, x2), "rsm_cd_fake_dynamo.onnx", dynamo=True, **EXPORT_KW)

fake forward OK -> output shape: (1, 1, 256, 256)

===== EXPORT ATTEMPT: torchscript -> rsm_cd_fake_torchscript.onnx =====


W0628 14:45:45.825000 58 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


/tmp/ipykernel_58/664370153.py:13: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(

RESULT: SUCCESS

===== EXPORT ATTEMPT: dynamo -> rsm_cd_fake_dynamo.onnx =====


W0628 14:45:46.339000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0628 14:45:46.341000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0628 14:45:46.343000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0628 14:45:46.345000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


nts violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
[torch.onnx] Obtain model graph for `RSM_CD([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `RSM_CD([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/us

{'ok': True, 'error': None}

In [24]:
# Compact summary table of all four attempts.
print(f"{'attempt':45s} | result")
print("-" * 70)
for label, r in results.items():
    status = "SUCCESS" if r["ok"] else f"FAILED ({r['error'][:60] if r['error'] else '?'})"
    print(f"{label:45s} | {status}")

attempt                                       | result
----------------------------------------------------------------------
torchscript -> rsm_cd_v3_torchscript.onnx     | SUCCESS
dynamo -> rsm_cd_v3_dynamo.onnx               | FAILED (TorchExportError: Failed to export the model with torch.expo)
torchscript -> rsm_cd_fake_torchscript.onnx   | SUCCESS
dynamo -> rsm_cd_fake_dynamo.onnx             | SUCCESS


## 4.5 Verify the exported graph — *did the scan actually cross the boundary?*

**Critical:** `torch.onnx.export` returning without raising does **not** mean the ONNX is valid
or correct. `SelectiveScanOflex` has no ONNX symbolic, so the legacy exporter (with
`autograd_inlining`) traces *into* its forward and hits the pybind C++ op
`selective_scan_cuda_oflex.fwd(...)`. The two likely outcomes both *look* like "success":

1. a **custom-domain op node** that onnxruntime / TensorRT cannot execute, or
2. the scan output **constant-folded** from the dummy input — the graph then returns a fixed
   tensor regardless of input.

This section turns "didn't raise" into a real yes/no via three checks on the v3 graph:
**(i)** `onnx.checker` + an op/domain inventory (any non-standard domain = a custom op a backend
must implement); **(ii)** does the output **change when the input changes** (catches constant
folding); **(iii)** does onnxruntime's output **match PyTorch** (catches a wrong/no-op scan).

In [25]:
import numpy as np
from collections import Counter
import onnx

V3_ONNX = "rsm_cd_v3_torchscript.onnx"   # the only attempt that produced a file

# (i) structural inventory --------------------------------------------------
m = onnx.load(V3_ONNX)
try:
    onnx.checker.check_model(m)
    print("onnx.checker: OK")
except Exception as e:
    print("onnx.checker FAILED:", repr(e)[:300])

nodes = m.graph.node
print("num nodes:", len(nodes))
print("op-type counts:", dict(Counter(n.op_type for n in nodes)))

domains = set(n.domain for n in nodes)
print("node domains:", domains)
custom = sorted({n.op_type for n in nodes if n.domain not in ("", "ai.onnx")})
print("NON-STANDARD-domain ops (a backend must implement these):", custom)

# Heuristic: a constant-folded scan shows up as the graph being dominated by Constant
# nodes and/or having very few real compute ops for a 51.95M-param model.
n_const = sum(1 for n in nodes if n.op_type == "Constant")
print(f"Constant nodes: {n_const} / {len(nodes)}")

onnx.checker: OK
num nodes: 9126
op-type counts: {'Identity': 184, 'Conv': 50, 'Transpose': 170, 'LayerNormalization': 100, 'Constant': 3701, 'Div': 62, 'Erf': 32, 'Add': 362, 'Mul': 274, 'MatMul': 120, 'Shape': 330, 'Gather': 390, 'Slice': 240, 'Sigmoid': 60, 'Unsqueeze': 1470, 'Concat': 457, 'Reshape': 390, 'Cast': 150, 'Range': 120, 'Mod': 60, 'ConstantOfShape': 120, 'Equal': 60, 'Where': 60, 'Expand': 60, 'ScatterElements': 60, 'Sub': 30, 'Relu': 9, 'Resize': 5}
node domains: {''}
NON-STANDARD-domain ops (a backend must implement these): []
Constant nodes: 3701 / 9126


In [26]:
# (ii) + (iii) behavioural checks ------------------------------------------
import onnxruntime as ort

try:
    sess = ort.InferenceSession(V3_ONNX, providers=["CPUExecutionProvider"])
    ort_ok = True
except Exception as e:
    ort_ok = False
    print("onnxruntime FAILED to load the model:", repr(e)[:400])
    print("=> the graph contains an op the runtime cannot execute (expected if the scan",
          "became a custom node).")

if ort_ok:
    def rnd():
        return np.random.randn(1, 3, 256, 256).astype(np.float32)
    a1, a2 = rnd(), rnd()
    b1, b2 = rnd(), rnd()
    out_a = sess.run(None, {"t1": a1, "t2": a2})[0]
    out_b = sess.run(None, {"t1": b1, "t2": b2})[0]

    changed = not np.allclose(out_a, out_b)
    print("(ii) output CHANGES when input changes:", changed,
          "  <- if False, the scan was constant-folded (export is hollow)")

    # (iii) correctness vs the live PyTorch v3 model on the SAME input
    with torch.no_grad():
        t_out = net(torch.from_numpy(a1).to(device),
                    torch.from_numpy(a2).to(device)).cpu().numpy()
    max_abs = float(np.abs(out_a - t_out).max())
    print(f"(iii) max|ORT - PyTorch| = {max_abs:.4g}  (CPU fp32 vs CUDA fp32)")
    print("      allclose(atol=1e-2):", np.allclose(out_a, t_out, atol=1e-2))

(ii) output CHANGES when input changes: True   <- if False, the scan was constant-folded (export is hollow)
(iii) max|ORT - PyTorch| = 0.003196  (CPU fp32 vs CUDA fp32)
      allclose(atol=1e-2): True


## 5. Finding & recommended path

**Headline (surprising, and better than the documented risk predicted): the naive PyTorch -> ONNX
export of `RSM_CD(v3)` already works via the *legacy TorchScript exporter*, and the result is a
fully-standard-op, numerically-correct ONNX graph. None of the three feared remediation paths
(plugin / decompose / custom-op) are needed for the export boundary.** The remaining Phase-4 risk
moves *downstream* to ONNX -> TensorRT ingestion.

### Export attempts (§3/§4)
| attempt | result |
|---|---|
| v3 / TorchScript (legacy) | **SUCCESS** — wrote `rsm_cd_v3_torchscript.onnx` (verified genuine, see below) |
| v3 / dynamo (torch.export) | **FAILED** — `RuntimeError: Cannot access data pointer of Tensor (FakeTensor)… erroneously tracing into a custom kernel… wrap the custom kernel into an opaque custom op`, raised at `rs_mamba_cd.py:201` (`selective_scan_cuda_oflex.fwd`). The canonical SSM/custom-CUDA export failure. |
| fake / TorchScript | SUCCESS |
| fake / dynamo | SUCCESS (modulo a harmless `Resize` opset-17 downgrade warning; graph kept at opset 18) |

### Verification of the v3 graph (§4.5) — the decisive evidence
- **onnx.checker:** OK
- **Nodes:** 9126, **all in the default domain `''`** → **zero custom-domain ops**. The scan +
  `CrossScan`/`CrossMerge` came out as *standard* ONNX ops (`Gather`/`ScatterElements`/`Range`/
  `Mod`/`Where`/`Expand`/`MatMul`/`Conv`/`LayerNormalization`/`Resize`…).
- **onnxruntime loads & runs:** yes (CPU EP).
- **(ii) output changes with input:** **True** → not constant-folded.
- **(iii) max|ORT − PyTorch| = 0.0032, allclose(atol=1e-2): True** on *fresh* random input →
  numerically correct. (Graph was traced with the original dummy yet matches on new input, so the
  SSM is genuinely live in the graph — not baked.)

### Why the two backends disagree
The **legacy** exporter traces with **real tensors**: it actually *runs* `selective_scan_cuda_oflex.fwd`
on CUDA during tracing, and the surrounding tensor algebra (cross-scan gather/scatter, einsums,
projections) is recorded as standard ops. The **dynamo/torch.export** path traces with **FakeTensors**,
which cannot enter the opaque CUDA kernel (no `data_ptr`) → it fails exactly at the scan and nowhere
else.

### Scan-only vs. broader blocker
**Scan is the SOLE blocker.** `fake/dynamo` SUCCEEDS while `v3/dynamo` fails *only* at
`selective_scan_cuda_oflex.fwd` → `CrossScan`/`CrossMerge` and the rest of the graph are fully
exportable (and indeed appear as standard ops in the v3 graph above).

### Recommended next-session path
**Path (0): proceed with the legacy-TorchScript ONNX and de-risk ONNX -> TensorRT next** — try to
build a TensorRT engine from `rsm_cd_v3_torchscript.onnx` (e.g. `trtexec --onnx=… --fp16` or the TRT
Python API). The PyTorch->ONNX problem is solved; the open question is whether TensorRT ingests this
9126-node graph. Paths (a)/(b)/(c) become relevant **only if TRT rejects it** — and the documented
fix for the dynamo failure ("wrap the custom kernel into an opaque custom op" via `torch.library`)
is the fallback, likely unnecessary.

### Open questions / risks for next session (do not over-trust the green result)
1. **ONNX -> TensorRT is untested.** onnxruntime-CPU accepting the graph ≠ TensorRT accepting it.
   The graph leans on dynamic-shape ops TRT can restrict: `Range` (120), `Mod` (60),
   `ScatterElements` (60), `Where` (60), `Shape` (330), `Resize` (5). This is now the real risk.
2. **Dynamic batch axis declared but NOT exercised** — both ORT runs in §4.5 (ii) were batch 1.
   Re-run with batch>1 to confirm the `batch` dynamic axis actually works before relying on it.
3. **Legacy exporter is deprecated** (PyTorch 2.9+ default is dynamo). It works on torch 2.10 today
   but is not future-proof; pin the toolchain or plan the opaque-custom-op migration.
4. **H/W specialized to 256.** Only the batch axis is dynamic; spatial size is baked to 256 — fine
   for this pipeline (256px tiles), but note it.
5. **fp16/INT8 numerics untested** — §4.5 validated fp32 only. Precision calibration is still ahead.


---
*Phase 4 de-risk — investigative notebook. No TensorRT engine produced; see §5 for the path decision.*